# Frequency-sensitive head ablation — Pythia

This notebook is the first causal screen. It reads the five candidate heads chosen
by the analysis notebook, ablates them on the **held-out compounds**, and compares
them with five random heads matched by layer.

It measures change in the next-token distribution using KL divergence. It does not
yet claim that answer accuracy changed.


## Setup

In [ ]:
# Cell 0: Environment Detection
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

In [ ]:
# Cell 1: Colab Only — Install pinned dependencies
# ⚠️ Restart runtime after running this cell, then skip to Cell 2
if IN_COLAB:
    %pip install -q transformer_lens==2.18.0
    %pip install -q numpy==1.26.4
    %pip install -q transformers==4.57.6

In [ ]:
# Cell 1a: Confirm Transformer Lens version
from importlib.metadata import version
print("TransformerLens version:", version("transformer-lens"))

In [ ]:
# Cell 1b: Environment check after session restart
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")

In [ ]:
# Cell 2: Project Root & Path Setup
if IN_COLAB:
    from google.colab import userdata
    token = userdata.get("GH_TMLR")

    repo_owner = "trishasalas"
    repo_name = "tmlr"
    repo_url = f"https://{token}@github.com/{repo_owner}/{repo_name}.git"

    PROJECT_ROOT = Path("/content") / repo_name

    if not PROJECT_ROOT.exists():
        !git clone {repo_url} {PROJECT_ROOT}
else:
    # Local: notebook lives in notebooks/, project root is one level up
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Cell 3: Imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    print(f"Added {PROJECT_ROOT} to sys.path")
    
import torch
import src
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils

# Device selection
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

In [ ]:
# Cell 5 - Model name variable
model_name = "pythia-160m"

In [ ]:
# Cell 6: Load Model
model = HookedTransformer.from_pretrained(f"EleutherAI/{model_name}")

print(f"Layers: {model.cfg.n_layers}")
print(f"Heads: {model.cfg.n_heads}")
print(f"Hidden size: {model.cfg.d_model}")
print(f"Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

In [ ]:
# Cell 7: Settings
if IN_COLAB:
    %pip install -q scipy

CONDITION = "natural"
RANDOM_SEED = 42

assert CONDITION in {"natural", "uniform"}


In [ ]:
# Cell 8: Load candidate heads and reconstruct the held-out compound split
import numpy as np
import pandas as pd
import yaml
from scipy.stats import spearmanr, wilcoxon
from IPython.display import display
from src.ablation_manifest import write_ablation_manifest
from src.qk_ov import ablate_heads_at_position

candidate_path = (
    PROJECT_ROOT / "results" / "analysis" /
    f"effective_binding_head_candidates_{CONDITION}.csv"
)
if not candidate_path.exists():
    raise FileNotFoundError(
        f"Missing {candidate_path}. Run effective-binding-analysis.ipynb first."
    )

candidates = pd.read_csv(candidate_path)
chosen = candidates[
    (candidates["family"] == "pythia")
    & (candidates["model"] == model_name)
].sort_values("candidate_rank")

if len(chosen) != 5:
    raise ValueError(f"Expected 5 candidate heads for {model_name}; found {len(chosen)}")

selected_heads = list(zip(chosen["layer"].astype(int), chosen["head"].astype(int)))

# Same deterministic split used by the analysis notebook.
frequency = pd.read_csv(PROJECT_ROOT / "results" / "frequency" / "frequency_table.csv")
all_compounds = sorted(frequency["compound"].unique())
rng = np.random.default_rng(RANDOM_SEED)
rng.shuffle(all_compounds)
test_compounds = set(all_compounds[len(all_compounds) // 2:])

# Random controls are matched by layer, so depth cannot explain a difference.
selected_set = set(selected_heads)
control_heads = []
for layer, selected_head in selected_heads:
    choices = [
        h for h in range(model.cfg.n_heads)
        if (layer, h) not in selected_set and (layer, h) not in control_heads
    ]
    control_heads.append((layer, int(rng.choice(choices))))

print("Selected heads:", selected_heads)
print("Control heads: ", control_heads)
print("Held-out compounds:", len(test_compounds))


In [ ]:
# Cell 9: Run held-out ablations
with open(PROJECT_ROOT / "data" / "binding" / "accessibility.yaml", "r") as f:
    compounds = yaml.safe_load(f)["compounds"]

compounds = [case for case in compounds if case["name"] in test_compounds]
rows = []

for i, case in enumerate(compounds):
    print(f"{i + 1:2d}/{len(compounds)}  {case['name']}")
    prompt = (
        case["prompt"] if CONDITION == "natural"
        else f"A {case['word1']} {case['word2']} is"
    )

    selected_result = ablate_heads_at_position(
        model, prompt, selected_heads, case["word2"]
    )
    control_result = ablate_heads_at_position(
        model, prompt, control_heads, case["word2"]
    )

    rows.append({
        "compound": case["name"],
        "model": model_name,
        "condition": CONDITION,
        "selected_heads": str(selected_heads),
        "control_heads": str(control_heads),
        "selected_kl": selected_result["kl_base_to_ablated"],
        "control_kl": control_result["kl_base_to_ablated"],
        "selected_minus_control_kl": (
            selected_result["kl_base_to_ablated"]
            - control_result["kl_base_to_ablated"]
        ),
        "selected_top_changed": (
            selected_result["baseline_top"][0][0]
            != selected_result["ablated_top"][0][0]
        ),
        "control_top_changed": (
            control_result["baseline_top"][0][0]
            != control_result["ablated_top"][0][0]
        ),
    })

results = pd.DataFrame(rows).merge(
    frequency[["compound", "bigram_count"]],
    on="compound", how="left", validate="one_to_one",
)
results["log_frequency"] = np.log1p(results["bigram_count"])

output_dir = PROJECT_ROOT / "results" / "ablation" / "frequency_heads" / model_name
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"{model_name}-{CONDITION}-heldout-ablation.csv"
results.to_csv(output_path, index=False)
manifest_path = write_ablation_manifest(
    project_root=PROJECT_ROOT,
    csv_path=output_path,
    candidate_path=candidate_path,
    notebook_path=PROJECT_ROOT / "notebooks" / "frequency-head-ablation-pythia.ipynb",
    intervention_code_path=PROJECT_ROOT / "src" / "qk_ov.py",
    model_metadata_manifest=(
        PROJECT_ROOT / "results" / "effective_binding" / "pythia" / model_name
        / CONDITION / f"{model_name}-{CONDITION}-effective-binding.md"
    ),
    model=model,
    random_seed=RANDOM_SEED,
)
print(f"\nSaved: {output_path}")
print(f"Manifest: {manifest_path}")
display(results.head())


In [ ]:
# Cell 10: Summarize the causal screen
selected_frequency = spearmanr(results["log_frequency"], results["selected_kl"])
control_frequency = spearmanr(results["log_frequency"], results["control_kl"])

try:
    paired = wilcoxon(results["selected_kl"], results["control_kl"])
    paired_text = f"W={paired.statistic:.3f}, p={paired.pvalue:.4g}"
except ValueError as error:
    paired_text = f"not available ({error})"

print("Selected-head KL vs frequency:")
print(f"  rho={selected_frequency.statistic:+.3f}, p={selected_frequency.pvalue:.4g}")
print("Control-head KL vs frequency:")
print(f"  rho={control_frequency.statistic:+.3f}, p={control_frequency.pvalue:.4g}")
print("Paired selected vs matched-control KL:")
print(f"  {paired_text}")
print()
print("Interpretation guide:")
print("- Selected KL larger for rare compounds: rarity-sensitive heads matter more there.")
print("- Selected KL no larger than control: the observed head signal is not load-bearing.")
print("- This screen measures distributional impact, not answer correctness.")


In [ ]:
# Optional: commit this model's ablation output.
import os
os.chdir(PROJECT_ROOT)
!git config user.email "trisha@trishasalas.com"
!git config user.name "Trisha Salas"
!git add results/ablation/frequency_heads/
!git commit -m "held-out frequency-head ablation: {model_name} {CONDITION}"
!git push


### Delete model and clear cache


In [ ]:
import gc
del model
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()
elif device == "mps":
    torch.mps.empty_cache()
print("Memory cleared")
